<!--nav--> [🗺 Learning path](README.md) · **24/41** · ◀ [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) · [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) ▶

# vLLM: PagedAttention, Continuous Batching & an OpenAI-Compatible Server

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/vLLM_High_Throughput_Serving.ipynb)

The [fundamentals notebook](./Serving_Fundamentals_KV_Cache_Batching.ipynb) showed *why* naive serving
wastes GPUs: fragmented KV memory, straggler batches, idle slots. **vLLM** is the open-source engine
that fixed all three and became the de-facto standard for self-hosted LLM serving (it's what most
"OpenAI-compatible endpoint" products run underneath).

| Part | What happens |
|---|---|
| **A** | Baseline: Hugging Face `generate` throughput on 64 prompts (so the comparison is honest) |
| **B** | Same 64 prompts through vLLM's offline `LLM` API — measure the gap |
| **C** | How PagedAttention actually works (the 5-minute version worth having) |
| **D** | Automatic **prefix caching**, measured |
| **E** | Launch a real **OpenAI-compatible server** in this Colab and load-test it with concurrent clients |

**Runs on:** free Colab **T4** · **GPU required.** On a T4 we use `dtype="half"` (no bf16 on Turing);
on A100/L4 everything below works unchanged and faster.

**Engine notes (2025-era vLLM):** everything here uses the **V1 engine** — a 2025 rewrite that made
continuous batching scheduling near-zero-overhead and turned on **chunked prefill** and
**prefix caching by default**. Tested with `vllm>=0.10`; the APIs used (`LLM`, `SamplingParams`,
`vllm serve`) are the stable, long-lived surface.

In [ ]:
# Install. vLLM ships its own CUDA kernels + a pinned torch — on Colab this takes ~2-4 min.
# If pip reports it changed torch versions: Runtime > Restart session, then re-run from here.
!pip install -q -U vllm openai

import torch, time, gc, os
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"     # keep the notebook readable
assert torch.cuda.is_available(), "vLLM needs a GPU — in Colab: Runtime > Change runtime type > T4"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} · {p.total_memory/1e9:.1f} GB · SM {p.major}.{p.minor}")
import vllm; print("vLLM:", vllm.__version__)

## Part A · The honest baseline: Hugging Face `generate`

64 varied prompts, batch-of-8 static batching (already generous to HF — sequential batch-1 would be
far worse). We run this **first** so vLLM inherits a clean GPU afterwards.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

TOPICS = ["the Roman aqueducts", "how TCP handshakes work", "the Krebs cycle", "black hole evaporation",
          "sourdough fermentation", "the Marshall Plan", "how vaccines train immunity", "plate tectonics",
          "the Byzantine generals problem", "photosynthesis", "the gold standard", "how GPS knows where you are",
          "the printing press", "monsoon seasons", "the halting problem", "coral reef ecosystems"]
PROMPTS = [f"Explain {t} to a curious high-schooler in about 100 words." for t in TOPICS] * 4  # 64 prompts

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to("cuda").eval()

chat = [tok.apply_chat_template([{"role":"user","content":p}], add_generation_prompt=True, tokenize=False)
        for p in PROMPTS]

torch.cuda.synchronize(); t0 = time.perf_counter(); hf_tokens = 0
with torch.no_grad():
    for i in range(0, len(chat), 8):                       # static batches of 8
        enc = tok(chat[i:i+8], return_tensors="pt", padding=True).to("cuda")
        out = hf_model.generate(**enc, max_new_tokens=128, do_sample=True, temperature=0.8,
                                top_p=0.95, pad_token_id=tok.eos_token_id)
        hf_tokens += int((out[:, enc.input_ids.shape[1]:] != tok.eos_token_id).sum())
torch.cuda.synchronize()
hf_time = time.perf_counter() - t0
print(f"HF static batching: {hf_tokens} tokens in {hf_time:.1f}s -> {hf_tokens/hf_time:.0f} tok/s")

# Hand the GPU back before vLLM starts
del hf_model, out, enc; gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory now: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## Part B · Same job, through vLLM

The offline `LLM` API is the simplest entry point: give it *all* the prompts at once and the engine's
continuous-batching scheduler figures out the rest — no manual batching, no padding, no stragglers.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL,
    dtype="half",                  # T4 = Turing = no bf16; use "auto" on A100/L4
    max_model_len=2048,            # cap context -> more KV blocks available
    gpu_memory_utilization=0.85,   # fraction of VRAM vLLM may claim (weights + KV block pool)
)

sp = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)
chat_prompts = [tok.apply_chat_template([{"role":"user","content":p}], add_generation_prompt=True, tokenize=False)
                for p in PROMPTS]

t0 = time.perf_counter()
outs = llm.generate(chat_prompts, sp)
vllm_time = time.perf_counter() - t0
vllm_tokens = sum(len(o.outputs[0].token_ids) for o in outs)

print(f"vLLM continuous batching: {vllm_tokens} tokens in {vllm_time:.1f}s -> {vllm_tokens/vllm_time:.0f} tok/s")
print(f"\n{'':>24}{'tokens/s':>10}")
print(f"{'HF static batch=8':>24}{hf_tokens/hf_time:>10.0f}")
print(f"{'vLLM (64 at once)':>24}{vllm_tokens/vllm_time:>10.0f}   <- {(vllm_tokens/vllm_time)/(hf_tokens/hf_time):.1f}x")
print("\nSample output:", outs[0].outputs[0].text[:200], "...")

**Where does the speedup come from?** Three stacked effects, in rough order of importance here:

1. **Continuous batching** — all 64 requests are in flight together; a finishing sequence's slot is
   refilled the same iteration. HF's static batches wait for each batch's straggler.
2. **No padding** — HF pads every sequence in a batch to the longest; vLLM's paged KV means sequences
   of different lengths coexist without padding tokens burning compute.
3. **Fused, purpose-built kernels** — paged attention kernels, fused sampling, CUDA graphs for decode
   (the V1 engine captures the decode step as a graph replay, eliminating Python overhead).

## Part C · PagedAttention — the idea that named the project

The fundamentals notebook showed classic serving pre-allocates one **contiguous** KV buffer per
request at `max_len` → 60–80% waste. vLLM's authors noticed this is *exactly* the problem operating
systems solved in the 1960s with **virtual memory**:

```
                OS virtual memory          PagedAttention
 unit           4KB page                   KV block (16 tokens' worth of K/V)
 mapping        page table per process     block table per sequence
 allocation     on page fault              a block at a time, as the sequence grows
 sharing        shared pages (fork, mmap)  shared prefix blocks + copy-on-write
```

Each sequence's KV cache becomes a list of *logical* blocks mapped to *physical* blocks scattered
anywhere in one big GPU pool:

```
                     physical KV block pool (all of GPU KV memory)
                   ┌────┬────┬────┬────┬────┬────┬────┬────┬────┐
                   │ #0 │ #1 │ #2 │ #3 │ #4 │ #5 │ #6 │ #7 │ #8 │ ...
                   └────┴────┴────┴────┴────┴────┴────┴────┴────┘
 seq A (37 toks)  block table: [#4, #0, #7]          ← 3 blocks, last one 5/16 full
 seq B (20 toks)  block table: [#2, #5]              ← grows a block ONLY when needed
```

Consequences:
- **Internal waste ≤ one block per sequence** (~16 tokens), vs up to `max_len` before. External
  fragmentation: zero — any free block fits any sequence.
- **More free memory = bigger batches** = the throughput you measured above.
- **Sharing is a pointer copy**: two sequences with the same prompt point at the same physical
  blocks (copy-on-write when they diverge). Parallel sampling (n=8) and beam search get ~55% memory
  sharing for free — and it's the mechanism behind Part D.

## Part D · Prefix caching — stop re-prefilling your system prompt

Real workloads repeat themselves: every request in a chatbot shares the system prompt; RAG shares
document chunks; agents re-send growing conversation histories. With paged KV, vLLM can hash each
full block and **keep it around after the request ends** — the next request whose prefix hashes to
the same blocks skips prefill for them entirely.

The V1 engine enables this **by default** (`enable_prefix_caching=True`). Watch it work:

In [ ]:
# One long shared system prompt + many different user questions - the classic chatbot shape.
SYSTEM = ("You are a meticulous research assistant for a maritime history museum. "
          "Answer in a formal tone, cite centuries not years, keep answers under 120 words, "
          "never speculate beyond the historical record, and always end by offering one related "
          "topic the visitor might explore next. ") * 20        # ~1100 tokens of shared prefix

QUESTIONS = ["Why did square sails give way to fore-and-aft rigs?",
             "What did ship's biscuit actually taste like?",
             "How were longitude problems solved at sea?",
             "What was life like for a cabin boy?",
             "How did steam power change naval strategy?",
             "Why were figureheads carved on ships?",
             "How did press gangs operate?",
             "What made the tea clippers so fast?"]

def run_round(label):
    prompts = [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
        add_generation_prompt=True, tokenize=False) for q in QUESTIONS]
    t0 = time.perf_counter()
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=64))
    dt = time.perf_counter() - t0
    n_prompt = sum(len(o.prompt_token_ids) for o in outs)
    print(f"{label}: {dt:.2f}s for {len(QUESTIONS)} requests ({n_prompt} prompt tokens)")
    return dt

cold = run_round("Round 1 (cold - full prefill of the shared prefix)")
warm = run_round("Round 2 (warm - shared prefix blocks served from cache)")
print(f"\nPrefix caching speedup on this shape: {cold/warm:.1f}x")
print("Round 1 already shares the prefix WITHIN the batch; round 2 also skips its prefill entirely.")
print("In production the win compounds: multi-turn chats re-send history every turn.")

**Rule of thumb:** put the *shared, stable* content (system prompt, few-shot examples, retrieved
documents) at the **front** of the prompt and per-request content at the end — prefix caching matches
block-by-block from the start, so one early differing token forfeits everything after it.

(SGLang, vLLM's main rival, generalized this idea into **RadixAttention** — a radix tree over all
cached prefixes, so *partially* overlapping prompts share too. vLLM's hash-based scheme now achieves
much of the same; more in [notebook 24](./Speculative_Decoding_Advanced_Serving.ipynb).)

## Part E · The real thing: an OpenAI-compatible server

Offline `LLM` is for batch jobs. Production serving is `vllm serve` — an HTTP server speaking the
OpenAI API (`/v1/chat/completions`, `/v1/completions`, `/v1/models`, plus Prometheus `/metrics`),
with the same engine underneath handling all clients concurrently. Any OpenAI SDK client works
against it by changing `base_url`.

First free the GPU from our offline engine, then launch:

In [ ]:
# The offline engine and the server can't both hold 85% of a 16GB T4 - release the former.
del llm; gc.collect(); torch.cuda.empty_cache()
print(f"GPU allocated after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

import subprocess
server = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--dtype", "half",
     "--max-model-len", "2048",
     "--gpu-memory-utilization", "0.85",
     "--max-num-seqs", "32",            # cap concurrent sequences (scheduler slots)
     "--port", "8000"],
    stdout=open("vllm_server.log", "w"), stderr=subprocess.STDOUT)

import urllib.request
print("Waiting for server (engine init + CUDA graph capture, ~1-3 min on T4)...")
for i in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print(f"Server up after ~{i*2}s"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Server didn't start - check vllm_server.log:\n" + open("vllm_server.log").read()[-3000:])

In [ ]:
# Any OpenAI client works - just point base_url at our GPU.
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

# 1) A plain chat completion
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In two sentences: why do ships float?"}],
    max_tokens=80, temperature=0.7)
print("Reply:", resp.choices[0].message.content)
print("Usage:", resp.usage.prompt_tokens, "prompt +", resp.usage.completion_tokens, "completion tokens")

# 2) Streaming - measure TTFT and inter-token latency like a real client would
t0 = time.perf_counter(); first = None; n = 0
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Tell a 100-word story about a lighthouse keeper."}],
    max_tokens=150, temperature=0.8, stream=True)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if first is None: first = time.perf_counter() - t0
        n += 1
total = time.perf_counter() - t0
print(f"\nStreaming: TTFT {first*1000:.0f}ms · {n} chunks in {total:.1f}s · TPOT {((total-first)/max(n-1,1))*1000:.0f}ms")

In [ ]:
# 3) Load test: 32 concurrent clients against one T4.
#    The server interleaves ALL of them in one continuously-batched decode loop.
from concurrent.futures import ThreadPoolExecutor

def one_request(i):
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"Give me exactly three fun facts about the number {i}."}],
        max_tokens=96, temperature=0.8)
    return time.perf_counter() - t0, r.usage.completion_tokens

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=32) as ex:
    results = list(ex.map(one_request, range(32)))
wall = time.perf_counter() - t0

lat = sorted(r[0] for r in results)
tokens = sum(r[1] for r in results)
print(f"32 concurrent requests: wall {wall:.1f}s · {tokens} tokens -> {tokens/wall:.0f} tok/s aggregate")
print(f"latency p50 {lat[15]:.1f}s · p95 {lat[30]:.1f}s (vs sequential estimate ~{sum(lat):.0f}s)")

# The server also exposes Prometheus metrics - this is what you'd graph in production:
metrics = urllib.request.urlopen("http://localhost:8000/metrics").read().decode()
for line in metrics.splitlines():
    if any(k in line for k in ("prompt_tokens_total", "generation_tokens_total",
                               "gpu_cache_usage_perc")) and not line.startswith("#"):
        print(line)

In [ ]:
# Cleanup - stop the server so the GPU is free for whatever you run next.
server.terminate(); server.wait(timeout=20)
print("Server stopped.")

## The knobs that matter in production

| Flag | What it trades |
|---|---|
| `--gpu-memory-utilization` | headroom vs KV pool size. Bigger pool → more concurrent sequences → more throughput |
| `--max-model-len` | max context vs KV blocks per request. Don't serve 128k if your users send 4k |
| `--max-num-seqs` | concurrency cap. Raise for throughput, lower to protect TPOT under load |
| `--max-num-batched-tokens` | chunked-prefill budget per step — how much prefill may squeeze in beside decode (V1 tunes this well by default) |
| `--quantization awq / gptq / fp8` | fit bigger models / free KV memory — measured in the [next notebook](./Quantized_Serving_Showdown.ipynb) |
| `--tensor-parallel-size N` | shard one big model across N GPUs (NVLink strongly preferred) |
| `--speculative-config ...` | draft-model / n-gram speculation — [notebook 24](./Speculative_Decoding_Advanced_Serving.ipynb) |
| `--kv-cache-dtype fp8` | halve KV memory on supported GPUs (Ada/Hopper) |

## Recap

- vLLM = **PagedAttention** (virtual memory for KV) + **continuous batching** (Orca-style
  iteration-level scheduling) + purpose-built kernels, behind an **OpenAI-compatible** HTTP API.
- You measured: multi-× throughput over HF static batching, prefix-cache speedup on shared prompts,
  and 32 concurrent clients on a free T4 with real p50/p95 numbers.
- The 2025 **V1 engine** made the good defaults automatic: chunked prefill, prefix caching,
  CUDA-graph decode, zero-overhead scheduling.

### Further reading
- [vLLM paper (SOSP '23)](https://arxiv.org/abs/2309.06180) · [vLLM V1 architecture blog](https://blog.vllm.ai/2025/01/27/v1-alpha-release.html)
- [vLLM docs — OpenAI-compatible server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html)
- [SGLang & RadixAttention](https://arxiv.org/abs/2312.07104) — the other engine you should know

▶ **Next:** [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) — AWQ vs GPTQ vs FP16:
memory, speed, and what you actually lose.